## Semantic Analysis-Based Sockpuppet Detection in Wikipedia

This section explores sockpuppet detection through semantic analysis, focusing on understanding and leveraging the meanings and relationships of words within the Wikipedia dataset. The method employs various NLP techniques to analyze the text data, aiming to identify patterns or anomalies indicative of sockpuppet behavior based solely on content semantics.


In [5]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import torch

# Function to encode the data into the format BERT understands
def encode_data(tokenizer, texts, max_length=512):
    return tokenizer(text=texts, add_special_tokens=True, max_length=max_length, truncation=True, padding='max_length', return_attention_mask=True, return_tensors='pt')

# Prepare the dataset for BERT
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Load the pre-split training and testing sets
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Using BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Encode the data
train_encodings = encode_data(tokenizer, train_data['edit_text'].tolist())
test_encodings = encode_data(tokenizer, test_data['edit_text'].tolist())

# Convert labels to list
y_train = train_data['is_sockpuppet'].tolist()
y_test = test_data['is_sockpuppet'].tolist()

# Convert to Dataset
train_dataset = Dataset(train_encodings, y_train)
test_dataset = Dataset(test_encodings, y_test)

# Load BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Setup training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()

# Print results
print("Evaluation results:", results)

              precision    recall  f1-score   support

           0       0.78      0.64      0.70      5379
           1       0.69      0.82      0.75      5381

    accuracy                           0.73     10760
   macro avg       0.73      0.73      0.73     10760
weighted avg       0.73      0.73      0.73     10760

Accuracy: 0.7274163568773234


## Sentiment Analysis-Based Sockpuppet Detection in Wikipedia

In this part of the notebook, sentiment analysis is used to detect sockpuppets on Wikipedia. The approach analyzes the emotional tone and subjective expressions in user contributions, looking for sentiment patterns that are often associated with deceptive or manipulative online behaviors characteristic of sockpuppets.


In [6]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset, RandomSampler
import pandas as pd
from textblob import TextBlob
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

# Load the pre-split training and testing sets
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Load pre-trained BERT tokenizer and BERT model for sequence classification
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Function to encode texts using BERT tokenizer
def encode_texts(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

# Encode the texts and prepare the data loader
def prepare_data(df):
    inputs = encode_texts(df['edit_text'].tolist())
    labels = torch.tensor(df['is_sockpuppet'].tolist())
    dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
    sampler = RandomSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=16)

train_loader = prepare_data(train_data)
test_loader = prepare_data(test_data)

# Define the training function
def train(model, loader):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    total_loss, total_accuracy = 0, 0

    for batch in tqdm(loader, desc="Training"):
        batch = tuple(b.to('cuda') for b in batch)  # Move to GPU
        inputs = {'input_ids': batch[0],
                  'attention_mask': batch[1],
                  'labels': batch[2]}
        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs[0]
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Training loss: {total_loss / len(loader)}")

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
train(model, train_loader)

# Function to evaluate the model
def evaluate(model, loader):
    model.eval()
    predictions, true_labels = [], []

    for batch in tqdm(loader, desc="Evaluating"):
        batch = tuple(b.to('cuda') for b in batch)  # Move to GPU
        inputs = {'input_ids': batch[0],
                  'attention_mask': batch[1]}
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs[0]
        predictions.extend(torch.argmax(logits, dim=-1).tolist())
        true_labels.extend(batch[2].tolist())

    print(classification_report(true_labels, predictions))
    print("Accuracy:", accuracy_score(true_labels, predictions))

# Evaluate the model
evaluate(model, test_loader)

              precision    recall  f1-score   support

           0       0.58      0.43      0.49      5379
           1       0.55      0.68      0.61      5381

    accuracy                           0.56     10760
   macro avg       0.56      0.56      0.55     10760
weighted avg       0.56      0.56      0.55     10760

Accuracy: 0.5567843866171004


## Integrated Semantic and Sentiment Analysis for Sockpuppet Detection in Wikipedia

This section combines semantic and sentiment analysis to enhance the accuracy of sockpuppet detection in Wikipedia. By integrating both analytical dimensions, the methodology seeks to provide a comprehensive view of the textual data, tapping into both the explicit meaning of the text (semantics) and the underlying emotions or attitudes (sentiment) to more effectively identify sockpuppet accounts.


In [7]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import time  # Import the time module

class SockpuppetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Load the data
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the data
train_encodings = tokenizer(train_data['edit_text'].tolist(), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_data['edit_text'].tolist(), truncation=True, padding=True, max_length=128)

# Create datasets
train_dataset = SockpuppetDataset(train_encodings, train_data['is_sockpuppet'].tolist())
test_dataset = SockpuppetDataset(test_encodings, test_data['is_sockpuppet'].tolist())

# Load pre-trained BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    report_to="all",                 # Ensure all logging outputs are reported
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Measure the time before training
start_time = time.time()

# Train the model
trainer.train()

# Calculate and print the training time
end_time = time.time()
total_training_time = end_time - start_time
print(f"Total training time: {total_training_time:.2f} seconds")

# Evaluate the model
results = trainer.evaluate()

print(results)

              precision    recall  f1-score   support

           0       0.77      0.63      0.70      5379
           1       0.69      0.81      0.75      5381

    accuracy                           0.72     10760
   macro avg       0.73      0.72      0.72     10760
weighted avg       0.73      0.72      0.72     10760

Accuracy: 0.7232342007434944
